# Data Quality and Reconciliation Report

Comprehensive DQ checks across Bronze, Silver, and Gold layers:
1. **Ingestion completeness** — all 8 bronze tables non-empty
2. **Primary key uniqueness** — gold dimensions and fact composite PK
3. **No null surrogate keys** — fact_sales FK columns
4. **Numeric measures** — quantity > 0, price > 0, discount 0–10%, tax 0–8%
5. **Gold vs Silver reconciliation** — revenue and row counts match
6. **Quarantine counts** — rejected records reported per table

Raises a **pipeline-stopped error** when any critical check fails.

In [0]:
%sql
-- Check 1: Ingestion Completeness — all 8 bronze source tables non-empty
SELECT 'customer'  AS table_name, COUNT(*) AS row_count,
  CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM tpch_dev.bronze.customer
UNION ALL
SELECT 'orders',    COUNT(*), CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.orders
UNION ALL
SELECT 'lineitem',  COUNT(*), CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.lineitem
UNION ALL
SELECT 'nation',   COUNT(*), CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.nation
UNION ALL
SELECT 'part',     COUNT(*), CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.part
UNION ALL
SELECT 'partsupp', COUNT(*), CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.partsupp
UNION ALL
SELECT 'region',   COUNT(*), CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.region
UNION ALL
SELECT 'supplier', COUNT(*), CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.supplier
ORDER BY table_name

In [0]:
%sql
-- Check 2: Primary keys populated and unique in Gold dimensions and fact
SELECT 'dim_geography' AS table_name, 'geography_key' AS pk_column,
  COUNT(*) AS total_rows, COUNT(DISTINCT geography_key) AS distinct_keys,
  COUNT(*) - COUNT(geography_key) AS null_keys,
  CASE WHEN COUNT(*) = COUNT(DISTINCT geography_key) AND COUNT(geography_key) = COUNT(*)
       THEN 'PASS' ELSE 'FAIL' END AS status
FROM tpch_dev.gold.dim_geography
UNION ALL
SELECT 'dim_customer', 'customer_key',
  COUNT(*), COUNT(DISTINCT customer_key),
  COUNT(*) - COUNT(customer_key),
  CASE WHEN COUNT(*) = COUNT(DISTINCT customer_key) AND COUNT(customer_key) = COUNT(*)
       THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.dim_customer
UNION ALL
SELECT 'dim_product', 'product_key',
  COUNT(*), COUNT(DISTINCT product_key),
  COUNT(*) - COUNT(product_key),
  CASE WHEN COUNT(*) = COUNT(DISTINCT product_key) AND COUNT(product_key) = COUNT(*)
       THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.dim_product
UNION ALL
SELECT 'dim_supplier', 'supplier_key',
  COUNT(*), COUNT(DISTINCT supplier_key),
  COUNT(*) - COUNT(supplier_key),
  CASE WHEN COUNT(*) = COUNT(DISTINCT supplier_key) AND COUNT(supplier_key) = COUNT(*)
       THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.dim_supplier
UNION ALL
SELECT 'fact_sales', 'order_key+line_number',
  COUNT(*), COUNT(DISTINCT CONCAT(order_key, '|', line_number)),
  (COUNT(*) - COUNT(order_key)) + (COUNT(*) - COUNT(line_number)),
  CASE WHEN COUNT(*) = COUNT(DISTINCT CONCAT(order_key, '|', line_number))
         AND COUNT(order_key) = COUNT(*) AND COUNT(line_number) = COUNT(*)
       THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.fact_sales
ORDER BY table_name

In [0]:
%sql
-- Check 3: fact_sales contains no null surrogate / foreign keys
SELECT 'customer_key' AS key_column,
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(customer_key) AS null_count,
  CASE WHEN COUNT(*) = COUNT(customer_key) THEN 'PASS' ELSE 'FAIL' END AS status
FROM tpch_dev.gold.fact_sales
UNION ALL
SELECT 'product_key', COUNT(*),
  COUNT(*) - COUNT(product_key),
  CASE WHEN COUNT(*) = COUNT(product_key) THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.fact_sales
UNION ALL
SELECT 'supplier_key', COUNT(*),
  COUNT(*) - COUNT(supplier_key),
  CASE WHEN COUNT(*) = COUNT(supplier_key) THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.fact_sales
UNION ALL
SELECT 'customer_nation_key', COUNT(*),
  COUNT(*) - COUNT(customer_nation_key),
  CASE WHEN COUNT(*) = COUNT(customer_nation_key) THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.fact_sales
UNION ALL
SELECT 'supplier_nation_key', COUNT(*),
  COUNT(*) - COUNT(supplier_nation_key),
  CASE WHEN COUNT(*) = COUNT(supplier_nation_key) THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.fact_sales
ORDER BY key_column

In [0]:
%sql
-- Check 4: Numeric measures satisfy documented business rules
SELECT 'quantity_positive' AS rule_name,
  SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) AS violations,
  CASE WHEN SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM tpch_dev.gold.fact_sales
UNION ALL
SELECT 'gross_amount_positive',
  SUM(CASE WHEN gross_amount <= 0 THEN 1 ELSE 0 END),
  CASE WHEN SUM(CASE WHEN gross_amount <= 0 THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.fact_sales
UNION ALL
SELECT 'discount_0_to_10_pct',
  SUM(CASE WHEN discount < 0 OR discount > 0.10 THEN 1 ELSE 0 END),
  CASE WHEN SUM(CASE WHEN discount < 0 OR discount > 0.10 THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.fact_sales
UNION ALL
SELECT 'tax_0_to_8_pct',
  SUM(CASE WHEN tax < 0 OR tax > 0.08 THEN 1 ELSE 0 END),
  CASE WHEN SUM(CASE WHEN tax < 0 OR tax > 0.08 THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.fact_sales
ORDER BY rule_name

In [0]:
%sql
-- Check 5: Gold revenue and row counts reconcile with Silver
WITH gold_stats AS (
  SELECT COUNT(*) AS gold_rows, ROUND(SUM(net_revenue), 2) AS gold_revenue
  FROM tpch_dev.gold.fact_sales
),
silver_stats AS (
  SELECT COUNT(*) AS silver_rows,
    ROUND(SUM(extended_price * (1 - discount) * (1 + tax)), 2) AS silver_revenue
  FROM tpch_dev.silver.lineitem
)
SELECT 'row_count' AS metric,
  gold_rows   AS gold_value,
  silver_rows  AS silver_value,
  gold_rows - silver_rows AS diff,
  CASE WHEN gold_rows = silver_rows THEN 'PASS' ELSE 'FAIL' END AS status
FROM gold_stats CROSS JOIN silver_stats
UNION ALL
SELECT 'revenue',
  gold_revenue, silver_revenue,
  ROUND(gold_revenue - silver_revenue, 2),
  CASE WHEN ABS(gold_revenue - silver_revenue) < 0.01 THEN 'PASS' ELSE 'FAIL' END
FROM gold_stats CROSS JOIN silver_stats

In [0]:
%sql
-- Check 6: Rejected records counted per quarantine table (informational)
SELECT 'region_quarantine'   AS quarantine_table, COUNT(*) AS rejected_count FROM tpch_dev.silver.region_quarantine
UNION ALL
SELECT 'nation_quarantine',    COUNT(*) FROM tpch_dev.silver.nation_quarantine
UNION ALL
SELECT 'customer_quarantine',  COUNT(*) FROM tpch_dev.silver.customer_quarantine
UNION ALL
SELECT 'supplier_quarantine', COUNT(*) FROM tpch_dev.silver.supplier_quarantine
UNION ALL
SELECT 'part_quarantine',      COUNT(*) FROM tpch_dev.silver.part_quarantine
UNION ALL
SELECT 'partsupp_quarantine',  COUNT(*) FROM tpch_dev.silver.partsupp_quarantine
UNION ALL
SELECT 'orders_quarantine',    COUNT(*) FROM tpch_dev.silver.orders_quarantine
UNION ALL
SELECT 'lineitem_quarantine',  COUNT(*) FROM tpch_dev.silver.lineitem_quarantine
ORDER BY quarantine_table

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Final Summary: Aggregate all critical DQ checks and raise error on any FAIL
# ─────────────────────────────────────────────────────────────────────────────

all_failures = spark.sql("""
WITH check1_ingestion AS (
  SELECT 'Check 1: Ingestion Completeness' AS check_name, table_name AS detail
  FROM (
    SELECT 'customer'  AS table_name, CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS status FROM tpch_dev.bronze.customer
    UNION ALL SELECT 'orders',    CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.orders
    UNION ALL SELECT 'lineitem',  CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.lineitem
    UNION ALL SELECT 'nation',   CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.nation
    UNION ALL SELECT 'part',     CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.part
    UNION ALL SELECT 'partsupp', CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.partsupp
    UNION ALL SELECT 'region',   CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.region
    UNION ALL SELECT 'supplier', CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END FROM tpch_dev.bronze.supplier
  ) WHERE status = 'FAIL'
),
check2_pk AS (
  SELECT 'Check 2: PK Uniqueness' AS check_name, table_name AS detail
  FROM (
    SELECT 'dim_geography' AS table_name,
      CASE WHEN COUNT(*) = COUNT(DISTINCT geography_key) AND COUNT(geography_key) = COUNT(*) THEN 'PASS' ELSE 'FAIL' END AS status
    FROM tpch_dev.gold.dim_geography
    UNION ALL
    SELECT 'dim_customer',
      CASE WHEN COUNT(*) = COUNT(DISTINCT customer_key) AND COUNT(customer_key) = COUNT(*) THEN 'PASS' ELSE 'FAIL' END
    FROM tpch_dev.gold.dim_customer
    UNION ALL
    SELECT 'dim_product',
      CASE WHEN COUNT(*) = COUNT(DISTINCT product_key) AND COUNT(product_key) = COUNT(*) THEN 'PASS' ELSE 'FAIL' END
    FROM tpch_dev.gold.dim_product
    UNION ALL
    SELECT 'dim_supplier',
      CASE WHEN COUNT(*) = COUNT(DISTINCT supplier_key) AND COUNT(supplier_key) = COUNT(*) THEN 'PASS' ELSE 'FAIL' END
    FROM tpch_dev.gold.dim_supplier
    UNION ALL
    SELECT 'fact_sales',
      CASE WHEN COUNT(*) = COUNT(DISTINCT CONCAT(order_key, '|', line_number))
             AND COUNT(order_key) = COUNT(*) AND COUNT(line_number) = COUNT(*)
           THEN 'PASS' ELSE 'FAIL' END
    FROM tpch_dev.gold.fact_sales
  ) WHERE status = 'FAIL'
),
check3_null_keys AS (
  SELECT 'Check 3: Null Surrogate Keys' AS check_name, key_column AS detail
  FROM (
    SELECT 'customer_key' AS key_column,
      CASE WHEN COUNT(*) = COUNT(customer_key) THEN 'PASS' ELSE 'FAIL' END AS status
    FROM tpch_dev.gold.fact_sales
    UNION ALL
    SELECT 'product_key', CASE WHEN COUNT(*) = COUNT(product_key) THEN 'PASS' ELSE 'FAIL' END
    FROM tpch_dev.gold.fact_sales
    UNION ALL
    SELECT 'supplier_key', CASE WHEN COUNT(*) = COUNT(supplier_key) THEN 'PASS' ELSE 'FAIL' END
    FROM tpch_dev.gold.fact_sales
    UNION ALL
    SELECT 'customer_nation_key', CASE WHEN COUNT(*) = COUNT(customer_nation_key) THEN 'PASS' ELSE 'FAIL' END
    FROM tpch_dev.gold.fact_sales
    UNION ALL
    SELECT 'supplier_nation_key', CASE WHEN COUNT(*) = COUNT(supplier_nation_key) THEN 'PASS' ELSE 'FAIL' END
    FROM tpch_dev.gold.fact_sales
  ) WHERE status = 'FAIL'
),
check4_numeric AS (
  SELECT 'Check 4: Numeric Measures' AS check_name, rule_name AS detail
  FROM (
    SELECT 'quantity_positive' AS rule_name,
      CASE WHEN SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END AS status
    FROM tpch_dev.gold.fact_sales
    UNION ALL
    SELECT 'gross_amount_positive',
      CASE WHEN SUM(CASE WHEN gross_amount <= 0 THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM tpch_dev.gold.fact_sales
    UNION ALL
    SELECT 'discount_0_to_10_pct',
      CASE WHEN SUM(CASE WHEN discount < 0 OR discount > 0.10 THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM tpch_dev.gold.fact_sales
    UNION ALL
    SELECT 'tax_0_to_8_pct',
      CASE WHEN SUM(CASE WHEN tax < 0 OR tax > 0.08 THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM tpch_dev.gold.fact_sales
  ) WHERE status = 'FAIL'
),
check5_reconciliation AS (
  SELECT 'Check 5: Gold vs Silver Reconciliation' AS check_name, metric AS detail
  FROM (
    SELECT 'row_count' AS metric,
      CASE WHEN (SELECT COUNT(*) FROM tpch_dev.gold.fact_sales) = (SELECT COUNT(*) FROM tpch_dev.silver.lineitem)
           THEN 'PASS' ELSE 'FAIL' END AS status
    UNION ALL
    SELECT 'revenue',
      CASE WHEN ABS(
        (SELECT ROUND(SUM(net_revenue), 2) FROM tpch_dev.gold.fact_sales) -
        (SELECT ROUND(SUM(extended_price * (1 - discount) * (1 + tax)), 2) FROM tpch_dev.silver.lineitem)
      ) < 0.01 THEN 'PASS' ELSE 'FAIL' END
  ) WHERE status = 'FAIL'
)
SELECT * FROM check1_ingestion
UNION ALL SELECT * FROM check2_pk
UNION ALL SELECT * FROM check3_null_keys
UNION ALL SELECT * FROM check4_numeric
UNION ALL SELECT * FROM check5_reconciliation
""")

failures = all_failures.collect()

if failures:
    print("=" * 60)
    print("  PIPELINE STOPPED — DATA QUALITY CHECKS FAILED")
    print("=" * 60)
    for f in failures:
        print(f"  FAIL: {f.check_name} — {f.detail}")
    print("=" * 60)
    raise Exception(f"Pipeline stopped: {len(failures)} critical DQ check(s) failed.")
else:
    print("=" * 60)
    print("  ALL DATA QUALITY CHECKS PASSED")
    print("=" * 60)